In [4]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn import datasets

digits = datasets.load_digits()
X  = digits.data.astype(np.float64)   # (1797, 64)

# 1. Padronizar (obrigatório)
X_std = StandardScaler().fit_transform(X)

# 2. Ajustar com todas as componentes para inspecionar a variância
pca_full = PCA(random_state=42).fit(X_std)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)

n90 = int(np.argmax(var_acum >= 0.90) + 1)
print(f"Componentes para 90% da variância: {n90} de {X.shape[1]}")

# 3. Reduzir de fato
pca = PCA(n_components=n90, random_state=42)
X_pca = pca.fit_transform(X_std)

# 4. Reconstruir (útil para medir perda e filtrar ruído)
X_rec = pca.inverse_transform(X_pca)

print(f"Shape original     : {X_std.shape}")
print(f"Shape reduzido     : {X_pca.shape}")
print(f"Variância retida   : {var_acum[n90-1]:.1%}")
print(f"Erro de reconstrução (MSE): {np.mean((X_rec - X_std)**2):.5f}")

Componentes para 90% da variância: 31 de 64
Shape original     : (1797, 64)
Shape reduzido     : (1797, 31)
Variância retida   : 90.0%
Erro de reconstrução (MSE): 0.09497


In [6]:
Xc = X_std - X_std.mean(axis=0)                       # centralizar
cov = np.cov(Xc, rowvar=False)                        # matriz de covariância
lambdas, V = np.linalg.eigh(cov)                      # autovalores/autovetores

idx = np.argsort(lambdas)[::-1]                       # ordem decrescente
lambdas, V = lambdas[idx], V[:, idx]

Z = Xc @ V[:, :n90]                                   # projeção
print(f"Variância explicada por PC1: {lambdas[0]/lambdas.sum():.2%}")

Variância explicada por PC1: 12.03%
